# Pengujian Model Serving API - Heart Disease Prediction
**Pengembang:** M. Rizal Basri  
**Berkas:** `rizalbasri-testing.ipynb` (Saran 3 Submission Dicoding)  

Notebook ini digunakan untuk menguji dan melakukan prediction request ke sistem machine learning yang dijalankan di cloud maupun lokal, serta memverifikasi keterbukaan metrik Prometheus.


## 1. Menentukan Target URL Endpoint
Menyiapkan URL endpoint API server (dapat diarahkan ke URL Cloud deployment seperti Heroku ataupun localhost).


In [1]:
import requests
import json
import time

# Ganti dengan URL Cloud Deployment Anda jika sudah di-deploy
BASE_URL = "http://localhost:8000"
print(f"Target API Endpoint: {BASE_URL}")


Target API Endpoint: http://localhost:8000


## 2. Pengujian Endpoint Root (`/`) dan Health Check (`/health`)


In [2]:
root_response = requests.get(f"{BASE_URL}/")
print("Root Endpoint Response:")
print(json.dumps(root_response.json(), indent=2))

health_response = requests.get(f"{BASE_URL}/health")
print("\nHealth Check Response:")
print(json.dumps(health_response.json(), indent=2))


Root Endpoint Response:
{
  "service": "Heart Disease Prediction ML Serving API",
  "author": "M. Rizal Basri",
  "pipeline": "rizalbasri-pipeline",
  "model_path": "serving_model_dir\\1789577747",
  "status": "online",
  "endpoints": {
    "health": "/health",
    "predict": "/predict (POST)",
    "metrics": "/metrics",
    "docs": "/docs"
  }
}

Health Check Response:
{
  "status": "healthy",
  "model_loaded": true
}


## 3. Pengujian Prediction Request (`POST /predict`)
Mengirimkan sampel data pasien dengan fitur klinis untuk memperoleh hasil inferensi model.


In [3]:
sample_request = {
    "inputs": [
        {
            "age": 63, "sex": 1, "cp": 3, "trestbps": 145, "chol": 233,
            "fbs": 1, "restecg": 0, "thalach": 150, "exang": 0,
            "oldpeak": 2.3, "slope": 0, "ca": 0, "thal": 1
        },
        {
            "age": 37, "sex": 0, "cp": 1, "trestbps": 120, "chol": 180,
            "fbs": 0, "restecg": 1, "thalach": 175, "exang": 0,
            "oldpeak": 0.0, "slope": 2, "ca": 0, "thal": 2
        }
    ]
}

pred_response = requests.post(f"{BASE_URL}/predict", json=sample_request)
print(f"Status Code: {pred_response.status_code}")
print("Hasil Prediksi:")
print(json.dumps(pred_response.json(), indent=2))


Status Code: 200
Hasil Prediksi:
{
  "predictions": [
    {
      "probability": 0.9496,
      "prediction": 1,
      "diagnosis": "Heart Disease Detected"
    },
    {
      "probability": 0.9999,
      "prediction": 1,
      "diagnosis": "Heart Disease Detected"
    }
  ]
}


## 4. Pengujian Prometheus Metrics Scrape Endpoint (`/metrics`)
Memverifikasi bahwa metrik monitoring telah bertambah setelah request inferensi dieksekusi.


In [4]:
metrics_response = requests.get(f"{BASE_URL}/metrics")
print(f"Metrics Status Code: {metrics_response.status_code}")

# Filter metrik relevan
relevant_metrics = [line for line in metrics_response.text.split("\n")
                    if line.startswith("http_requests_total") or
                       line.startswith("model_predictions_total") or
                       line.startswith("model_loaded_status")]
print("Cuplikan Metrik Prometheus:")
for m in relevant_metrics[:10]:
    print(m)


Metrics Status Code: 200
Cuplikan Metrik Prometheus:
http_requests_total{endpoint="/health",http_status="200",method="GET"} 1.0
http_requests_total{endpoint="/predict",http_status="200",method="POST"} 1.0
model_predictions_total{prediction_label="Heart Disease Detected"} 2.0
model_loaded_status 1.0
